# 03 — LLM Knowledge Extraction
Uses the local LLM to mine training pairs from real Examples, Book chapters, and Proposals.
All generated ARO code is validated with `aro check`. Valid pairs are saved to
`knowledge_pairs.jsonl`.

Run **notebook 04** after this to warm-start fine-tune the model on these pairs
before the RL explore loop in notebook 06.

**Input:**  `../data/02_knowledge/knowledge.json`, `../../Examples/`, `../../Book/`
**Output:** `../data/02_knowledge/knowledge_pairs.jsonl`

In [1]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import json, re, sys, random, subprocess, tempfile
from pathlib import Path
from collections import Counter

DATA_OUT    = DATA_ROOT / '02_knowledge'

with open(DATA_IN / 'knowledge.json') as f:
    kb = json.load(f)

print(f'ARO root: {ARO_ROOT}')
print(f'Knowledge base: {len(kb["actions"])} actions, {len(kb["examples"])} examples, {len(kb["proposals"])} proposals')
print(f'Loading {MODEL_ID}...')
model, tokenizer, _, mlx_generate, make_sampler = load_model()
print('Model ready.')

Model ready.
Model ready.


In [2]:
# Build system prompt from action metadata
action_lines = []
for a in kb['actions']:
    if a['verbs']:
        v = '/'.join(a['verbs'][:3])
        p = ', '.join(a['prepositions'][:4])
        action_lines.append(f'- {v}  (role: {a["role"]}, prepositions: {p})')

SYSTEM_PROMPT = """You are an expert ARO language programmer.
ARO (Action Result Object) is a DSL for expressing business logic as natural-language statements.

SYNTAX:
  (FeatureSetName: BusinessActivity) {
      Verb [the] <result[:qualifier]> preposition [the] <object[:qualifier]>.
  }

KEY RULES:
- Articles (a/an/the) are optional everywhere
- String concatenation: ++ (NOT +)
- Variable names: hyphenated lowercase e.g. <user-id>, <order-total>
- Forbidden variable name prefixes: is-, with-, empty-
- Variables are IMMUTABLE — bind once; use a new name for each transformation
- Exactly ONE Application-Start per application (error if 0 or multiple)
- openapi.yaml REQUIRED for HTTP server; operationId must match feature set name
- Long-running apps: Keepalive the <application> for the <events>.

HTTP:
- Path params:   Extract the <id> from the <pathParameters: id>.
- Request body:  Extract the <data> from the <request: body>.
- Return statuses: <OK: status>, <Created: status>, <NoContent: status>,
                   <NotFound: status>, <BadRequest: status>, <Conflict: status>,
                   <Unauthorized: status>, <InternalServerError: status>

CONTROL FLOW:
- Conditional:   when <condition> { statements }
- Loop:          For each <item> in <list> { statements }
- Match:         match <var> { case value { statements } case other { statements } }
- Guard on declaration: (Name: EventName Handler) when <field> = value { ... }

COMPUTE & ARITHMETIC:
- Compute the <total> from <price> * <qty>.
- Compute the <upper: uppercase> from <text>.
- Compute the <len: length> from <text>.
- Supported ops: +, -, *, /, % (integers); ++ (strings)

CROSS-FEATURE SHARING:
- Publish as <alias> <variable>.  (makes variable accessible across feature sets in same business activity)

EVENTS:
- Emit a <EventName: event> with <data>.
- Handler:  (HandlerName: EventName Handler) { ... }
- State:    Accept the <new-state: toState> for the <entity>.

LIFECYCLE:
- Application-Start: required entry point
- Application-End: Success (optional graceful shutdown)
- Application-End: Error (optional crash handler)

AVAILABLE ACTIONS (verb → role → prepositions):
- Extract     REQUEST   from/with     — pull data from request/event/object
- Retrieve    REQUEST   from/where    — fetch from repository or service
- Fetch       REQUEST   from/with     — HTTP GET external URL
- Parse       REQUEST   from          — parse JSON/YAML/HTML/CSV text
- Request     REQUEST   from/with     — HTTP POST/PUT/PATCH
- Compute     OWN       from/with/for — arithmetic, string ops, built-in transforms
- Transform   OWN       from/with     — type coercion (e.g. string → int)
- Validate    OWN       for/with      — check constraints, return bool
- Compare     OWN       against/with  — compare two values, return bool
- Create      OWN       with          — instantiate struct or entity
- Merge       OWN       with          — combine objects or collections
- Filter      OWN       where/with    — select subset of collection
- Sort        OWN       by/with       — order a collection
- Split       OWN       by/from       — split string or list by delimiter
- Join        OWN       with/from     — join strings or list elements
- Accept      OWN       with/for      — state machine transition
- Return      RESPONSE  with/for      — send HTTP response and end feature set
- Throw       RESPONSE  with/for      — raise exception (propagates as error)
- Store       EXPORT    into          — persist to repository (auto-generates id)
- Update      EXPORT    into          — update existing record in repository
- Delete      EXPORT    from          — remove record from repository
- Log         EXPORT    to            — write to console or log service
- Send        EXPORT    to/with       — send email or message
- Emit        EXPORT    with          — publish domain event to EventBus
- Publish     EXPORT    as            — share variable across feature sets
- Start       EXPORT    with          — start a service (HTTP server, file monitor)
- Stop        EXPORT    with          — stop a running service
- Keepalive   EXPORT    for           — block until shutdown signal (for servers)
- Render      EXPORT    with/using    — render Mustache template
- Notify      EXPORT    with          — send notification to user(s)
- Configure   EXPORT    with          — set runtime configuration option
- Read        REQUEST   from          — read file contents
- Write       EXPORT    to            — write data to file
- Copy        EXPORT    to            — copy file to destination
- Move        EXPORT    to            — move file to destination

Always wrap ARO code in ```aro ... ``` fences.
Always generate complete, valid ARO that would pass `aro check`.
"""


def chat(messages, max_tokens=800):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return mlx_generate(
        model, tokenizer, prompt=prompt,
        max_tokens=max_tokens, verbose=False,
        sampler=make_sampler(temp=0.3),
    )


def extract_aro_blocks(text):
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]


def aro_check(code):
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            return r.returncode == 0, (r.stderr or r.stdout).strip()[:300]
    except FileNotFoundError:
        return None, 'aro_not_found'
    except subprocess.TimeoutExpired:
        return False, 'timeout'


def aro_run_quick(code, timeout=8):
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'run', tmp], capture_output=True, text=True, timeout=timeout)
            return r.returncode == 0, (r.stdout + r.stderr).strip()[:200]
    except FileNotFoundError:
        return None, 'aro_not_found'
    except subprocess.TimeoutExpired:
        return True, 'server_timeout'


# Resume support
_done_keys = set()
_pair_count = 0
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            if line.strip():
                try:
                    s = json.loads(line)
                    _done_keys.add((s['source'], s['instruction'][:80]))
                    _pair_count += 1
                except Exception:
                    pass
    print(f'Resuming — {_pair_count} pairs already saved')


def save_pair(source, instruction, output, score=1.0, metadata=None):
    key = (source, instruction[:80])
    if key in _done_keys:
        return False
    record = {
        'instruction': instruction,
        'output': output,
        'source': source,
        'score': score,
        'metadata': metadata or {},
        'notebook': 'NB03',
    }
    save_notebook_pair('NB03', record)
    _done_keys.add(key)
    return True


print(f'System prompt: {len(SYSTEM_PROMPT)} chars')
print('Helpers ready.')

Resuming — 3 pairs already saved
System prompt: 4733 chars
Helpers ready.


## Phase 1 — Real Examples → Training Pairs

For each of the 103 real, validated ARO examples:
- Ask the LLM to write a natural language instruction describing what the application does
- The code itself is already valid (taken verbatim from the repository)
- Result: (instruction, real_code) pairs with score=1.0

This is the highest-quality training data because the code is guaranteed correct.

In [3]:
clean_notebook_pairs('NB03')

def load_app_dir(app_path):
    """Load a single ARO application directory into the same dict shape as kb['examples']."""
    p = Path(app_path)
    aro_files = {}
    for f in sorted(p.rglob('*.aro')):
        rel = str(f.relative_to(p))
        aro_files[rel] = f.read_text()

    def read_opt(name):
        fp = p / name
        return fp.read_text() if fp.exists() else None

    return {
        'name':      p.name,
        'dir':       str(p),
        'aro_files': aro_files,
        'openapi':   read_opt('openapi.yaml'),
        'readme':    read_opt('README.md'),
        'expected':  None,
    }


# ── Collect all examples ─────────────────────────────────────────────────────
# 1. Examples already in the knowledge base (from ./Examples/ and any previously
#    discovered paths in notebook 01/02)
all_examples = list(kb['examples'])

# 2. Directly scan ARO-Application for multi-file production apps
ARO_APP_DIR = (ARO_ROOT.parent / 'ARO-Application').resolve()
if ARO_APP_DIR.exists():
    known_names = {e['name'] for e in all_examples}
    for app_subdir in sorted(ARO_APP_DIR.iterdir()):
        if not app_subdir.is_dir():
            continue
        if app_subdir.name in known_names:
            continue            # already in kb['examples'] — skip duplicate
        if not list(app_subdir.rglob('*.aro')):
            continue            # no ARO files
        all_examples.append(load_app_dir(app_subdir))
        print(f'  Added ARO-Application/{app_subdir.name}', flush=True)
else:
    print(f'  ARO-Application not found at {ARO_APP_DIR}')

print(f'\nTotal examples for Phase 1: {len(all_examples)} '
      f'({len(kb["examples"])} from knowledge base + '
      f'{len(all_examples) - len(kb["examples"])} from ARO-Application)')


# ── Generate instruction for each example ────────────────────────────────────
print(f'\n--- Phase 1: Real Examples → Training Pairs ---')
count = 0

for ex in all_examples:
    if not ex['aro_files']:
        continue

    source_key = f'example:{ex["name"]}'

    # Build a compact code representation for the LLM context window
    parts = []
    if ex.get('openapi'):
        parts.append(f'## openapi.yaml\n```yaml\n{ex["openapi"][:600]}\n```')
    for fname, code in list(ex['aro_files'].items())[:5]:   # cap at 5 files
        parts.append(f'## {fname}\n```aro\n{code[:1200]}\n```')
    full_repr = '\n\n'.join(parts)[:4000]

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': (
            f'Here is a complete ARO application called "{ex["name"]}":\n\n{full_repr}\n\n'
            f'Write ONE clear, specific natural language instruction (2-3 sentences) that '
            f'describes exactly what this application does — specific enough that a developer '
            f'following the instruction would produce this code.\n\n'
            f'Reply with ONLY the instruction text. No markdown, no labels.'
        )},
    ]
    instruction = chat(messages, max_tokens=200).strip()
    instruction = re.sub(r'^(#+\s*|Instruction:\s*|\*\*[^*]+\*\*:\s*)', '', instruction, flags=re.MULTILINE).strip()
    instruction = instruction.split('\n')[0].strip()

    if not instruction or len(instruction) < 20:
        continue

    # Reconstruct full output (all .aro files + openapi)
    output_parts = []
    if ex.get('openapi'):
        output_parts.append(f'## openapi.yaml\n```yaml\n{ex["openapi"]}\n```')
    for fname, code in ex['aro_files'].items():
        output_parts.append(f'## {fname}\n```aro\n{code}\n```')
    output = '\n\n'.join(output_parts)

    if save_pair(source_key, instruction, output, score=1.0,
                 metadata={'example': ex['name'], 'has_openapi': bool(ex.get('openapi')),
                            'source_dir': ex.get('dir', '')}):
        count += 1
        tag = ' [ARO-App]' if ex['name'] in {e.name for e in ARO_APP_DIR.iterdir() if ARO_APP_DIR.exists() and e.is_dir()} else ''
        print(f'  [{count}]{tag} {ex["name"]}: {instruction[:85]}', flush=True)

print(f'\nPhase 1 done: {count} new pairs from real examples')


Phase 1 done: 110 new pairs from real examples


## Phase 2 — Book: TheLanguageGuide → Training Pairs

For each of the 47 Language Guide chapters:
- Read the chapter markdown
- Ask the LLM to generate 3 instruction→code pairs that demonstrate concepts in that chapter
- Validate each generated code block with `aro check`
- Save only validated pairs

The Language Guide covers every aspect of ARO syntax, so this phase teaches the model
the full breadth of the language.

In [4]:
print(f'\n--- Phase 2: TheLanguageGuide → Training Pairs ---')
count = 0

guide_dir = ARO_ROOT / 'Book' / 'TheLanguageGuide'
chapters = sorted(guide_dir.glob('*.md'))   # chapters + appendices

for chapter_path in chapters:
    chapter_name = chapter_path.stem
    content = chapter_path.read_text()

    # Only process chapters with ARO code blocks
    code_blocks = extract_aro_blocks(content)
    if not code_blocks:
        print(f'  {chapter_name}: no ARO blocks, skipping', flush=True)
        continue

    content_trimmed = content[:4500]

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': (
            f'Here is "{chapter_name}" from the ARO Language Guide:\n\n{content_trimmed}\n\n'
            f'Generate 3 distinct instruction → ARO code training pairs based on this chapter.\n'
            f'Each pair must demonstrate a DIFFERENT concept from the chapter.\n'
            f'The ARO code must be complete and use only syntax shown in the chapter.\n\n'
            f'Format EXACTLY like this:\n\n'
            f'### Pair 1\n'
            f'**Instruction:** <natural language instruction>\n'
            f'**Code:**\n'
            f'```aro\n<complete valid ARO code>\n```\n\n'
            f'### Pair 2\n'
            f'**Instruction:** ...\n'
            f'**Code:**\n'
            f'```aro\n...\n```\n\n'
            f'### Pair 3\n'
            f'**Instruction:** ...\n'
            f'**Code:**\n'
            f'```aro\n...\n```'
        )},
    ]

    output_text = chat(messages, max_tokens=1400)

    # Parse pairs from output
    pair_blocks = re.split(r'###\s*Pair\s*\d+', output_text)
    chapter_count = 0

    for block in pair_blocks[1:]:
        # Extract instruction
        instr_match = re.search(r'\*\*Instruction:\*\*\s*(.+?)(?=\n\*\*|\n```|\Z)', block, re.DOTALL)
        if not instr_match:
            instr_match = re.search(r'Instruction:\s*(.+?)(?=\n(?:Code:|```|\*\*)|\Z)', block, re.DOTALL)
        if not instr_match:
            continue
        instruction = instr_match.group(1).strip().replace('\n', ' ')

        codes = extract_aro_blocks(block)
        if not codes or not instruction or len(instruction) < 15:
            continue

        code = codes[0]
        check_ok, check_err = aro_check(code)
        if check_ok is False:
            short_err = check_err.split('\n')[0][:70]
            print(f'    x {chapter_name}: {short_err}', flush=True)
            continue

        score = 1.0 if check_ok is True else 0.8  # 0.8 if aro not available
        if save_pair(f'book:{chapter_name}', instruction, f'```aro\n{code}\n```',
                     score=score, metadata={'chapter': chapter_name}):
            chapter_count += 1
            count += 1
            print(f'  [{count}] {chapter_name}: {instruction[:80]}', flush=True)

    if chapter_count == 0:
        print(f'  {chapter_name}: 0 valid pairs extracted', flush=True)

print(f'\nPhase 2 done: {count} new pairs from Language Guide')


Phase 2 done: 38 new pairs from Language Guide


## Phase 3 — Book: AROByExample → Training Pairs

AROByExample builds a complete web crawler in ARO across 14 chapters.
Each chapter shows real, working ARO code for a specific concern (fetching, parsing, storing, etc.).
The LLM generates 2 training pairs per chapter from the complete worked examples.

In [5]:
print(f'\n--- Phase 3: AROByExample → Training Pairs ---')
count = 0

aro_by_example_dir = ARO_ROOT / 'Book' / 'AROByExample'
chapters = sorted(aro_by_example_dir.glob('Chapter*.md'))

for chapter_path in chapters:
    chapter_name = chapter_path.stem
    content = chapter_path.read_text()
    code_blocks = extract_aro_blocks(content)
    if not code_blocks:
        continue

    content_trimmed = content[:4000]

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': (
            f'Here is "{chapter_name}" from "ARO By Example" (a hands-on ARO tutorial):\n\n'
            f'{content_trimmed}\n\n'
            f'Generate 2 instruction → ARO code pairs from the examples in this chapter.\n\n'
            f'### Pair 1\n'
            f'**Instruction:** <what this code does>\n'
            f'**Code:**\n'
            f'```aro\n<the ARO code>\n```\n\n'
            f'### Pair 2\n'
            f'**Instruction:** ...\n'
            f'**Code:**\n'
            f'```aro\n...\n```'
        )},
    ]

    output_text = chat(messages, max_tokens=1000)
    pair_blocks = re.split(r'###\s*Pair\s*\d+', output_text)

    for block in pair_blocks[1:]:
        instr_match = re.search(r'\*\*Instruction:\*\*\s*(.+?)(?=\n\*\*|\n```|\Z)', block, re.DOTALL)
        if not instr_match:
            continue
        instruction = instr_match.group(1).strip().replace('\n', ' ')
        codes = extract_aro_blocks(block)
        if not codes or len(instruction) < 15:
            continue

        code = codes[0]
        check_ok, check_err = aro_check(code)
        if check_ok is False:
            continue

        if save_pair(f'aro_by_example:{chapter_name}', instruction,
                     f'```aro\n{code}\n```',
                     score=1.0 if check_ok is True else 0.8,
                     metadata={'chapter': chapter_name}):
            count += 1
            print(f'  [{count}] {chapter_name}: {instruction[:80]}', flush=True)

print(f'\nPhase 3 done: {count} new pairs from AROByExample')


Phase 3 done: 16 new pairs from AROByExample


## Phase 4 — Proposals → Contextualized Training Pairs

The language proposals contain hundreds of valid ARO code blocks that demonstrate specific
language features. The LLM provides a natural language instruction for each code block,
turning them into (instruction, validated_code) training pairs.

Only code that passes `aro check` is saved.

In [6]:
print(f'\n--- Phase 4: Proposals → Contextualized Pairs ---')
count = 0

for prop in kb['proposals']:
    prop_name = prop['name']

    # ── Build a code→(heading, body) lookup from qa_seeds for rich context ──
    seed_context = {}   # code_prefix → (heading, body)
    for seed in prop.get('qa_seeds', []):
        for code in seed.get('code_examples', []):
            if code.strip():
                seed_context[code.strip()[:60]] = (seed['heading'], seed['body'][:500])

    # ── Use ALL code blocks from the proposal (not just top 8) ───────────────
    # Also mine per-seed code_examples in case they differ from top-level blocks
    all_code_blocks = []
    seen_code = set()
    for code in prop.get('aro_code_blocks', []):
        norm = code.strip()
        if norm and norm not in seen_code:
            seen_code.add(norm)
            all_code_blocks.append(norm)
    for seed in prop.get('qa_seeds', []):
        for code in seed.get('code_examples', []):
            norm = code.strip()
            if norm and norm not in seen_code:
                seen_code.add(norm)
                all_code_blocks.append(norm)

    if not all_code_blocks:
        continue

    # Cap per proposal so one large spec doesn't dominate
    for i, code in enumerate(all_code_blocks[:15]):
        if len(code.strip()) < 40:
            continue

        code_key = f'proposal:{prop_name}:{i}'

        # Validate first — skip invalid snippets
        check_ok, check_err = aro_check(code)
        if check_ok is False:
            continue

        # Rich context from matching seed heading, or first seed body
        ctx_key = code.strip()[:60]
        if ctx_key in seed_context:
            heading, body = seed_context[ctx_key]
            context = f'Section "{heading}": {body}'
        elif prop.get('qa_seeds'):
            seed = prop['qa_seeds'][0]
            context = f'Section "{seed["heading"]}": {seed["body"][:300]}'
        else:
            context = ''

        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': (
                f'This ARO code example is from specification {prop_name}.\n\n'
                f'Context: {context}\n\n'
                f'```aro\n{code}\n```\n\n'
                f'Write ONE natural language instruction (1-2 sentences) that would '
                f'prompt a developer to write exactly this code. '
                f'Reply with ONLY the instruction text.'
            )},
        ]

        instruction = chat(messages, max_tokens=120).strip()
        instruction = re.sub(r'^(#+\s*|Instruction:\s*|\*\*[^*]+\*\*:\s*)', '', instruction).strip()
        instruction = instruction.split('\n')[0].strip()

        if not instruction or len(instruction) < 15:
            continue

        if save_pair(code_key, instruction, f'```aro\n{code}\n```',
                     score=1.0 if check_ok is True else 0.7,
                     metadata={'proposal': prop_name, 'block_index': i}):
            count += 1
            if count % 20 == 0:
                print(f'  [{count}] {prop_name}[{i}]: {instruction[:70]}', flush=True)

print(f'\nPhase 4 done: {count} new pairs from proposals')


Phase 4 done: 83 new pairs from proposals


## Summary

All pairs written to `knowledge_pairs.jsonl`.

**Next steps:**
- Run **notebook 04** — warm-start fine-tune on these pairs (+ any extras from notebook 05)
- Run **notebook 05** — generate more Q&A pairs from all Book chapters

In [7]:
all_pairs = []
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            if line.strip():
                try:
                    all_pairs.append(json.loads(line))
                except Exception:
                    pass

sources = Counter(p.get('source', 'unknown').split(':')[0] for p in all_pairs)
scores  = Counter(round(p.get('score', 1.0), 1) for p in all_pairs)

print(f'\nTotal knowledge pairs: {len(all_pairs)}')
print('\nBy source:')
for src, n in sorted(sources.items(), key=lambda x: -x[1]):
    print(f'  {src:30s}: {n}')
print('\nBy score:')
for score, n in sorted(scores.items(), key=lambda x: -x[0]):
    print(f'  {score}: {n}')


Total knowledge pairs: 2796

By source:
  unknown                       : 141
  example                       : 110
  proposal                      : 83
  aro/Book/Book/TheLanguageGuide/Chapter47-TerminalUI.md@0f958770aa: 58
  book                          : 38
  aro/Book/Book/TheLanguageGuide/Chapter16-HandlerGuards.md@0f958770aa: 32
  aro/Book/Book/TheLanguageGuide/Chapter11-Immutability.md@0f958770aa: 26
  aro/Book/Book/TheLanguageGuide/Chapter13-EventBus.md@0f958770aa: 26
  aro/Book/Book/TheLanguageGuide/Chapter26-Plugins.md@0f958770aa: 26
  aro/Book/Book/TheLanguageGuide/Chapter44-Templates.md@0f958770aa: 26
  aro/Book/Book/TheLanguageGuide/Chapter15-CustomEvents.md@0f958770aa: 24
  aro/Book/Book/TheLanguageGuide/Chapter27-NativeCompilation.md@0f958770aa: 24
  aro/Book/Book/TheLanguageGuide/Chapter42-Dates.md@0f958770aa: 24
  aro/Book/Book/TheLanguageGuide/Chapter23-Modules.md@9fc22b05fe: 24
  aro/Proposals/Proposals/ARO-0012-events-reactive.md@37e5c391a5: 24
  aro/Book/Book/TheL

In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
from pathlib import Path
from collections import Counter
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '03_llm_knowledge_extraction.png'

# Aggregate by readable source category + score bucket
_src_map = {
    'example':      'Phase 1 · Real examples',
    'book':         'Phase 2 · Language Guide',
    'aro_by_example': 'Phase 3 · AROByExample',
    'proposal':     'Phase 4 · Proposals',
}
_buckets = {v: {'score_1': 0, 'score_08': 0, 'score_07': 0} for v in _src_map.values()}
_other   = {'score_1': 0, 'score_08': 0, 'score_07': 0}

for p in all_pairs:
    raw_src = p.get('source', 'unknown').split(':')[0]
    bucket  = _src_map.get(raw_src, None)
    s       = round(p.get('score', 1.0), 1)
    target  = _buckets[bucket] if bucket else _other
    if s >= 1.0:   target['score_1']  += 1
    elif s >= 0.8: target['score_08'] += 1
    else:          target['score_07'] += 1

_categories = list(_src_map.values())
_s1  = [_buckets[c]['score_1']  for c in _categories]
_s08 = [_buckets[c]['score_08'] for c in _categories]
_s07 = [_buckets[c]['score_07'] for c in _categories]
_totals = [a + b + c for a, b, c in zip(_s1, _s08, _s07)]

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(_categories))
b1 = ax.bar(x, _s1,  label='score 1.0 (aro check ✓)', color='#2ecc71', edgecolor='white')
b2 = ax.bar(x, _s08, bottom=_s1, label='score 0.8 (unchecked)', color='#f39c12', edgecolor='white')
b3 = ax.bar(x, _s07, bottom=[a+b for a,b in zip(_s1,_s08)], label='score 0.7', color='#e74c3c', edgecolor='white')
for i, (tot, cat) in enumerate(zip(_totals, _categories)):
    ax.text(i, tot + 1, str(tot), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_xticks(list(x))
ax.set_xticklabels(_categories, fontsize=9)
ax.set_ylabel('Pairs')
ax.set_title(f'LLM Knowledge Extraction — {sum(_totals):,} pairs by phase and quality score',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, max(_totals, default=1) * 1.2)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')

Saved: run/2026-03-29/03_llm_knowledge_extraction.png
